# Clase 4: Aprendizaje Bayesiano & Naïve Bayes (CBS) 🔮📊

Este notebook interactivo acompaña el contenido teórico de la **Clase 4** del curso de **Aprendizaje Automático (FING - UdelaR)**.

### Contenidos:
1. **Teorema de Bayes e Inferencia MAP vs ML** (ejemplo clásico de test de dopaje / médico).
2. **Clasificador Bayesiano Sencillo (Naïve Bayes)**: Suposición de independencia condicional.
3. **Suavizado de Probabilidades**: Frecuencias relativas, $m$-estimador y corrección de Laplace.
4. **Inferencia en Escala Logarítmica** y cálculo de probabilidades a posteriori normalizadas.
5. **Visualización Gráfica**: Comparativa de distribuciones a priori vs a posteriori.
6. **Clasificador Bayesiano Óptimo (BOC)**: Ensamble probabilístico de hipótesis.

In [ ]:
# Importar módulos del paquete local 'algoritmos'
import sys
import os
sys.path.append(os.path.abspath('../../..'))

import pandas as pd
from algoritmos import (
    NaiveBayesClassifier,
    BayesOptimalClassifier,
    print_bayesian_trace,
    plot_prior_and_posterior
)
print("✅ Módulos importados correctamente.")

--- 
## 1. Ejemplo 1: Test de Dopaje con Inferencia Bayesiana Pura

Sabemos que el $1\%$ de los deportistas consumen sustancias prohibidas ($P(h) = 0{,}01$).
- Si consume ($h$): el test da positivo con probabilidad $90\%$ ($P(\oplus \mid h) = 0{,}90$).
- Si no consume ($\neg h$): el test da negativo con probabilidad $85\%$ ($P(\ominus \mid \neg h) = 0{,}85 \implies P(\oplus \mid \neg h) = 0{,}15$).

In [ ]:
P_h = 0.01
P_not_h = 0.99
P_pos_given_h = 0.90
P_pos_given_not_h = 0.15

# Probabilidad total de test positivo P(⊕)
P_pos = P_pos_given_h * P_h + P_pos_given_not_h * P_not_h

# Probabilidad a posteriori P(h | ⊕)
P_h_given_pos = (P_pos_given_h * P_h) / P_pos

print(f"P(⊕) = {P_pos:.4f}")
print(f"P(Dopaje | Test Positivo) = {P_h_given_pos:.4%}")

--- 
## 2. Dataset: Predicción de Críticas de Películas (Examen 2025)

In [ ]:
movies_data = [
    {'Genero': 'accion',          'Director': 'Nolan',      'Actor': 'Hathaway', 'Critica': 'Buena'},
    {'Genero': 'ciencia ficcion', 'Director': 'Villeneuve', 'Actor': 'Damon',    'Critica': 'Buena'},
    {'Genero': 'drama',           'Director': 'Nolan',      'Actor': 'Damon',    'Critica': 'Regular'},
    {'Genero': 'ciencia ficcion', 'Director': 'Villeneuve', 'Actor': 'Ferguson', 'Critica': 'Regular'},
    {'Genero': 'drama',           'Director': 'Scott',      'Actor': 'Ferguson', 'Critica': 'Buena'},
    {'Genero': 'accion',          'Director': 'Scott',      'Actor': 'Hathaway', 'Critica': 'Mala'},
]

df_movies = pd.DataFrame(movies_data)
df_movies

--- 
## 3. Entrenamiento con Naïve Bayes (CBS) y Traza Explicativa

In [ ]:
features = ['Genero', 'Director', 'Actor']
target = 'Critica'

# Entrenar clasificador con m-estimador m=0 (estimación de máxima verosimilitud)
nb_cbs = NaiveBayesClassifier(m=0.0).fit(movies_data, target_attr=target, features=features)

# Consulta #7 a clasificar: <ciencia ficcion, Nolan, Ferguson>
instancia_test = {'Genero': 'ciencia ficcion', 'Director': 'Nolan', 'Actor': 'Ferguson'}

# Imprimir desglose explicativo
print_bayesian_trace(nb_cbs, instancia_test)

In [ ]:
# Gráfico comparativo de probabilidades a priori vs a posteriori
plot_prior_and_posterior(nb_cbs, instancia_test, title="Distribución A Priori vs A Posteriori (Película Consulta)")

--- 
## 4. Comparación: Efecto del Suavizado con $m$-estimador ($m=1.0$)

In [ ]:
# Entrenar con m=1.0 (evita probabilidades nulas cuando un valor no aparece con una clase)
nb_m1 = NaiveBayesClassifier(m=1.0).fit(movies_data, target_attr=target, features=features)
print_bayesian_trace(nb_m1, instancia_test)

--- 
## 5. Clasificador Bayesiano Óptimo (BOC)

In [ ]:
# Combinación de 3 hipótesis de clasificación con sus respectivas probabilidades a posteriori
class RuleHypothesis:
    def __init__(self, mapping):
        self.mapping = mapping
    def predict_one(self, x):
        return self.mapping.get(x['Genero'], 'Buena')

h1 = RuleHypothesis({'accion': 'Mala', 'ciencia ficcion': 'Buena', 'drama': 'Regular'})
h2 = RuleHypothesis({'accion': 'Mala', 'ciencia ficcion': 'Regular', 'drama': 'Buena'})
h3 = RuleHypothesis({'accion': 'Buena', 'ciencia ficcion': 'Regular', 'drama': 'Regular'})

boc = BayesOptimalClassifier(
    hypotheses=[h1, h2, h3],
    hypothesis_posteriors=[0.4, 0.4, 0.2]
)

print(f"Predicción Óptima combinada: '{boc.predict_one(instancia_test)}'")
print(f"Probabilidades ponderadas: {boc.predict_proba_one(instancia_test)}")